In [6]:

from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split




# 

In [7]:
# ---- DATA: a deliberately NOISY dataset (label noise + 35 useless features) ----
X, y = make_classification(n_samples=600, n_features=40, n_informative=5,
                           n_redundant=0, n_repeated=0, flip_y=0.15,
                           class_sep=0.7, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.4, random_state=0, stratify=y)
scaler = StandardScaler().fit(X_train)
X_train_s, X_test_s = scaler.transform(X_train), scaler.transform(X_test)

In [8]:
big = dict(hidden_layer_sizes=(200, 200), max_iter=1000, random_state=0)

# 1) no early stopping — the big net MEMORISES the training set
plain = MLPClassifier(**big).fit(X_train_s, y_train)
tr, te = plain.score(X_train_s, y_train), plain.score(X_test_s, y_test)
print("No early stopping:")
print(f"  train acc {tr:.3f} | test acc {te:.3f}   (gap {tr - te:.3f}  <- overfitting!)")


No early stopping:
  train acc 1.000 | test acc 0.771   (gap 0.229  <- overfitting!)


In [9]:
stopped = MLPClassifier(early_stopping=True, validation_fraction=0.2,
                        n_iter_no_change=15, **big).fit(X_train_s, y_train)
tr2, te2 = stopped.score(X_train_s, y_train), stopped.score(X_test_s, y_test)
print("With early stopping:")
print(f"  train acc {tr2:.3f} | test acc {te2:.3f}   (gap {tr2 - te2:.3f})")
print(f"  stopped after {stopped.n_iter_} steps instead of 1000")

print("\nPerfect on train, weak on test = memorising, not learning. Early stopping")
print("shrinks the gap and generalises better with far less training (Day 14).")


With early stopping:
  train acc 0.936 | test acc 0.775   (gap 0.161)
  stopped after 40 steps instead of 1000

Perfect on train, weak on test = memorising, not learning. Early stopping
shrinks the gap and generalises better with far less training (Day 14).


In [10]:
alphas = [0.0001, 0.01, 1.0]

for alpha in alphas:
    mlp = MLPClassifier(
        alpha=alpha,
        **big
    )

    mlp.fit(X_train_s, y_train)

    train_acc = mlp.score(X_train_s, y_train)
    test_acc = mlp.score(X_test_s, y_test)

    print(f"alpha={alpha} | train={train_acc:.3f} | test={test_acc:.3f}")

alpha=0.0001 | train=1.000 | test=0.771
alpha=0.01 | train=1.000 | test=0.771
alpha=1.0 | train=1.000 | test=0.779
